In [1]:
import pandas as pd

In [2]:


# Load all files
studentInfo = pd.read_csv("studentInfo.csv")
studentReg = pd.read_csv("studentRegistration.csv")
studentAssess = pd.read_csv("studentAssessment.csv")
courses = pd.read_csv("courses.csv")
assessments = pd.read_csv("assessments.csv")

# Quick summary of each
for name, df in [("studentInfo", studentInfo), 
                  ("studentRegistration", studentReg),
                  ("studentAssessment", studentAssess),
                  ("courses", courses),
                  ("assessments", assessments)]:
    print(f"{'='*40}")
    print(f"FILE: {name}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print()
# ```

# ---

# But also — let's move them to the right folder properly. In VS Code terminal run:
# ```
# move *.csv data\raw\

FILE: studentInfo
Shape: (32593, 12)
Columns: ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result']

FILE: studentRegistration
Shape: (32593, 5)
Columns: ['code_module', 'code_presentation', 'id_student', 'date_registration', 'date_unregistration']

FILE: studentAssessment
Shape: (173912, 5)
Columns: ['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']

FILE: courses
Shape: (22, 3)
Columns: ['code_module', 'code_presentation', 'module_presentation_length']

FILE: assessments
Shape: (206, 6)
Columns: ['code_module', 'code_presentation', 'id_assessment', 'assessment_type', 'date', 'weight']



In [3]:
# Most important column — your target variable
print("FINAL RESULT DISTRIBUTION:")
print(studentInfo['final_result'].value_counts())
print()
print(studentInfo['final_result'].value_counts(normalize=True).mul(100).round(1))

print("\nGENDER DISTRIBUTION:")
print(studentInfo['gender'].value_counts())

print("\nAGE DISTRIBUTION:")
print(studentInfo['age_band'].value_counts())

print("\nTOP MODULES:")
print(studentInfo['code_module'].value_counts())

print("\nNULL VALUES IN studentInfo:")
print(studentInfo.isnull().sum())

FINAL RESULT DISTRIBUTION:
final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64

final_result
Pass           37.9
Withdrawn      31.2
Fail           21.6
Distinction     9.3
Name: proportion, dtype: float64

GENDER DISTRIBUTION:
gender
M    17875
F    14718
Name: count, dtype: int64

AGE DISTRIBUTION:
age_band
0-35     22944
35-55     9433
55<=       216
Name: count, dtype: int64

TOP MODULES:
code_module
BBB    7909
FFF    7762
DDD    6272
CCC    4434
EEE    2934
GGG    2534
AAA     748
Name: count, dtype: int64

NULL VALUES IN studentInfo:
code_module                0
code_presentation          0
id_student                 0
gender                     0
region                     0
highest_education          0
imd_band                1111
age_band                   0
num_of_prev_attempts       0
studied_credits            0
disability                 0
final_result               0
dtype: int64


In [4]:
import numpy as np

In [6]:
# ── 1. Create Churn Label ────────────────────────────────────
# Withdrawn = churned, Pass/Distinction = successful
# We'll exclude Fail for now — they're a separate segment
studentInfo['churned'] = studentInfo['final_result'].map({
    'Withdrawn': 1,
    'Pass': 0,
    'Distinction': 0,
    'Fail': 0  # we treat fail as not churned for now
})

print("CHURN DISTRIBUTION:")
print(studentInfo['churned'].value_counts())
print()

# ── 2. Handle imd_band nulls ─────────────────────────────────
# Fill missing imd_band with 'Unknown'
studentInfo['imd_band'] = studentInfo['imd_band'].fillna('Unknown')
print("imd_band nulls after fix:", studentInfo['imd_band'].isnull().sum())

# ── 3. Encode categorical columns for ML later ───────────────
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

cat_cols = ['gender', 'region', 'highest_education', 
            'imd_band', 'age_band', 'disability']

studentInfo_encoded = studentInfo.copy()
for col in cat_cols:
    studentInfo_encoded[col + '_encoded'] = le.fit_transform(studentInfo[col])

print("\nEncoded columns added successfully!")

# ── 4. Merge registration dates ──────────────────────────────
studentInfo_encoded = studentInfo_encoded.merge(
    studentReg[['id_student', 'code_module', 'code_presentation',
                'date_registration', 'date_unregistration']],
    on=['id_student', 'code_module', 'code_presentation'],
    how='left'
)

print("\nAfter merging registration data:")
print(studentInfo_encoded.shape)

# ── 5. Create early dropout flag ─────────────────────────────
# If unregistration date exists = dropped out
studentInfo_encoded['dropped_out'] = studentInfo_encoded['date_unregistration'].notnull().astype(int)
print("\nDropped out students:", studentInfo_encoded['dropped_out'].sum())

# ── 6. Merge assessment features ─────────────────────────────
# Get average score and number of assessments per student
assess_features = studentAssess.groupby('id_student').agg(
    avg_score=('score', 'mean'),
    num_assessments=('id_assessment', 'count'),
    avg_submission_day=('date_submitted', 'mean')
).reset_index()

studentInfo_encoded = studentInfo_encoded.merge(
    assess_features, on='id_student', how='left'
)

print("\nAfter merging assessment features:")
print(studentInfo_encoded.shape)
print("\nNull values in final dataset:")
print(studentInfo_encoded.isnull().sum())

# ── 7. Save cleaned data ─────────────────────────────────────
studentInfo_encoded.to_csv("cleaned.csv", index=False)
print("\n✅ Cleaned data saved!")


CHURN DISTRIBUTION:
churned
0    22437
1    10156
Name: count, dtype: int64

imd_band nulls after fix: 0

Encoded columns added successfully!

After merging registration data:
(32593, 21)

Dropped out students: 10072

After merging assessment features:
(32593, 25)

Null values in final dataset:
code_module                      0
code_presentation                0
id_student                       0
gender                           0
region                           0
highest_education                0
imd_band                         0
age_band                         0
num_of_prev_attempts             0
studied_credits                  0
disability                       0
final_result                     0
churned                          0
gender_encoded                   0
region_encoded                   0
highest_education_encoded        0
imd_band_encoded                 0
age_band_encoded                 0
disability_encoded               0
date_registration               45
date

In [8]:
# Fix remaining nulls

# 1. Registration date — fill with 0 (course start day)
studentInfo_encoded['date_registration'] = studentInfo_encoded['date_registration'].fillna(0)

# 2. Assessment nulls — students who never submitted anything
# Fill with -1 to flag them as "never engaged"
studentInfo_encoded['avg_score'] = studentInfo_encoded['avg_score'].fillna(-1)
studentInfo_encoded['num_assessments'] = studentInfo_encoded['num_assessments'].fillna(0)
studentInfo_encoded['avg_submission_day'] = studentInfo_encoded['avg_submission_day'].fillna(-1)

# 3. Create a powerful new feature — "never submitted" flag
studentInfo_encoded['never_submitted'] = (studentInfo_encoded['num_assessments'] == 0).astype(int)

print("Never submitted any assessment:", studentInfo_encoded['never_submitted'].sum())
print("Of those, how many churned?")
print(studentInfo_encoded[studentInfo_encoded['never_submitted']==1]['churned'].value_counts())

print("\nNull values remaining:")
print(studentInfo_encoded.isnull().sum().sum(), "total nulls")

# Save again
studentInfo_encoded.to_csv("cleaned.csv", index=False)
print("✅ Updated cleaned data saved!")

Never submitted any assessment: 5847
Of those, how many churned?
churned
1    4648
0    1199
Name: count, dtype: int64

Null values remaining:
22521 total nulls
✅ Updated cleaned data saved!


In [9]:
# Confirm remaining nulls are ONLY date_unregistration
print(studentInfo_encoded.isnull().sum()[studentInfo_encoded.isnull().sum() > 0])

date_unregistration    22521
dtype: int64
